In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "CMAPSSData"

columns = (
    ["unit_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train = pd.read_csv(
    data_dir / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

test = pd.read_csv(
    data_dir / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

test_rul = pd.read_csv(
    data_dir / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["RUL"]
)

# Training target
train["max_cycle"] = train.groupby("unit_id")["cycle"].transform("max")
train["RUL"] = train["max_cycle"] - train["cycle"]

RUL_CAP = 125
train["RUL_capped"] = train["RUL"].clip(upper=RUL_CAP)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train engines:", train["unit_id"].nunique())
print("Test engines:", test["unit_id"].nunique())
print("Target range:", train["RUL_capped"].min(), "-", train["RUL_capped"].max())

Train shape: (20631, 29)
Test shape: (13096, 26)
Train engines: 100
Test engines: 100
Target range: 0 - 125


In [2]:
from sklearn.model_selection import GroupShuffleSplit

candidate_features = (
    ["cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train,
        groups=train["unit_id"]
    )
)

train_fold = train.iloc[train_idx].copy()
val_fold = train.iloc[val_idx].copy()

# Sabit feature kontrolünü yalnızca training fold üzerinde yapıyoruz.
constant_features = [
    col for col in candidate_features
    if train_fold[col].nunique() == 1
]

feature_cols = [
    col for col in candidate_features
    if col not in constant_features
]

X_train = train_fold[feature_cols]
y_train = train_fold["RUL_capped"]

X_val = val_fold[feature_cols]
y_val = val_fold["RUL_capped"]

train_engine_ids = set(train_fold["unit_id"].unique())
val_engine_ids = set(val_fold["unit_id"].unique())

print("Constant features:", constant_features)
print("Number of model features:", len(feature_cols))
print("Training rows:", len(train_fold))
print("Validation rows:", len(val_fold))
print("Training engines:", len(train_engine_ids))
print("Validation engines:", len(val_engine_ids))
print(
    "Engines appearing in both groups:",
    train_engine_ids.intersection(val_engine_ids)
)

Constant features: ['setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Number of model features: 18
Training rows: 16561
Validation rows: 4070
Training engines: 80
Validation engines: 20
Engines appearing in both groups: set()


In [3]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)

train_dummy_pred = dummy_model.predict(X_train)
val_dummy_pred = dummy_model.predict(X_val)

train_mae = mean_absolute_error(y_train, train_dummy_pred)
train_rmse = mean_squared_error(y_train, train_dummy_pred) ** 0.5
train_r2 = r2_score(y_train, train_dummy_pred)

val_mae = mean_absolute_error(y_val, val_dummy_pred)
val_rmse = mean_squared_error(y_val, val_dummy_pred) ** 0.5
val_r2 = r2_score(y_val, val_dummy_pred)

print("Constant prediction:", dummy_model.constant_[0])
print("\nTraining metrics")
print(f"MAE:  {train_mae:.2f}")
print(f"RMSE: {train_rmse:.2f}")
print(f"R²:   {train_r2:.3f}")

print("\nValidation metrics")
print(f"MAE:  {val_mae:.2f}")
print(f"RMSE: {val_rmse:.2f}")
print(f"R²:   {val_r2:.3f}")

Constant prediction: [86.95881891]

Training metrics
MAE:  36.95
RMSE: 41.66
R²:   0.000

Validation metrics
MAE:  36.93
RMSE: 41.72
R²:   -0.000


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)

train_linear_pred = linear_model.predict(X_train)
val_linear_pred = linear_model.predict(X_val)

linear_train_mae = mean_absolute_error(y_train, train_linear_pred)
linear_train_rmse = mean_squared_error(y_train, train_linear_pred) ** 0.5
linear_train_r2 = r2_score(y_train, train_linear_pred)

linear_val_mae = mean_absolute_error(y_val, val_linear_pred)
linear_val_rmse = mean_squared_error(y_val, val_linear_pred) ** 0.5
linear_val_r2 = r2_score(y_val, val_linear_pred)

comparison = pd.DataFrame(
    {
        "Dummy": [
            train_mae,
            train_rmse,
            train_r2,
            val_mae,
            val_rmse,
            val_r2
        ],
        "Linear Regression": [
            linear_train_mae,
            linear_train_rmse,
            linear_train_r2,
            linear_val_mae,
            linear_val_rmse,
            linear_val_r2
        ]
    },
    index=[
        "Train MAE",
        "Train RMSE",
        "Train R²",
        "Validation MAE",
        "Validation RMSE",
        "Validation R²"
    ]
)

print(comparison.round(3))

print("\nValidation prediction range:")
print(
    f"{val_linear_pred.min():.2f} to "
    f"{val_linear_pred.max():.2f}"
)

                  Dummy  Linear Regression
Train MAE        36.949             16.679
Train RMSE       41.661             20.458
Train R²          0.000              0.759
Validation MAE   36.930             14.477
Validation RMSE  41.721             17.902
Validation R²    -0.000              0.816

Validation prediction range:
-29.83 to 156.24


In [5]:
import numpy as np

val_linear_pred_clipped = np.clip(
    val_linear_pred,
    0,
    RUL_CAP
)

clipping_comparison = pd.DataFrame({
    "Raw predictions": [
        mean_absolute_error(y_val, val_linear_pred),
        mean_squared_error(y_val, val_linear_pred) ** 0.5,
        r2_score(y_val, val_linear_pred)
    ],
    "Clipped predictions": [
        mean_absolute_error(y_val, val_linear_pred_clipped),
        mean_squared_error(y_val, val_linear_pred_clipped) ** 0.5,
        r2_score(y_val, val_linear_pred_clipped)
    ]
}, index=["Validation MAE", "Validation RMSE", "Validation R²"])

print(clipping_comparison.round(3))

print("\nPredictions below 0:", (val_linear_pred < 0).sum())
print("Predictions above 125:", (val_linear_pred > RUL_CAP).sum())
print(
    "Clipped prediction range:",
    val_linear_pred_clipped.min(),
    "to",
    val_linear_pred_clipped.max()
)

                 Raw predictions  Clipped predictions
Validation MAE            14.477               13.142
Validation RMSE           17.902               17.363
Validation R²              0.816                0.827

Predictions below 0: 71
Predictions above 125: 529
Clipped prediction range: 0.0 to 125.0


In [6]:
val_errors = pd.DataFrame({
    "actual_RUL": y_val.to_numpy(),
    "predicted_RUL": val_linear_pred_clipped
})

val_errors["absolute_error"] = (
    val_errors["actual_RUL"] - val_errors["predicted_RUL"]
).abs()

val_errors["RUL_range"] = pd.cut(
    val_errors["actual_RUL"],
    bins=[-1, 25, 50, 75, 100, 125],
    labels=["0–25", "26–50", "51–75", "76–100", "101–125"]
)

error_by_range = (
    val_errors.groupby("RUL_range", observed=True)["absolute_error"]
    .agg(rows="count", MAE="mean")
)

print(error_by_range.round(2))
print("\nTotal validation rows:", error_by_range["rows"].sum())

           rows    MAE
RUL_range             
0–25        520  13.41
26–50       500  17.99
51–75       500  15.16
76–100      500  10.74
101–125    2050  11.98

Total validation rows: 4070


In [7]:
val_errors["signed_error"] = (
    val_errors["predicted_RUL"] - val_errors["actual_RUL"]
)

val_errors["overestimated"] = val_errors["signed_error"] > 0

bias_by_range = val_errors.groupby(
    "RUL_range", observed=True
).agg(
    rows=("signed_error", "size"),
    mean_signed_error=("signed_error", "mean"),
    overestimate_rate=("overestimated", "mean")
)

bias_by_range["overestimate_pct"] = (
    bias_by_range["overestimate_rate"] * 100
)

print(
    bias_by_range[
        ["rows", "mean_signed_error", "overestimate_pct"]
    ].round(2)
)

           rows  mean_signed_error  overestimate_pct
RUL_range                                           
0–25        520              10.93             73.65
26–50       500              17.09             90.40
51–75       500              12.71             83.60
76–100      500               2.21             58.20
101–125    2050             -11.46              4.20


In [8]:
val_failure_cycles = val_fold.groupby("unit_id")["cycle"].max()

cutoff_by_engine = (
    val_failure_cycles * 2 / 3
).astype(int)

validation_snapshot = val_fold.loc[
    val_fold["cycle"].eq(
        val_fold["unit_id"].map(cutoff_by_engine)
    )
].sort_values("unit_id").copy()

print("Snapshot rows:", len(validation_snapshot))
print("Unique engines:", validation_snapshot["unit_id"].nunique())

print(
    validation_snapshot[
        ["unit_id", "cycle", "max_cycle", "RUL", "RUL_capped"]
    ].head()
)

Snapshot rows: 20
Unique engines: 20
      unit_id  cycle  max_cycle  RUL  RUL_capped
127         1    128        192   64          64
1025        5    179        269   90          90
2295       11    160        240   80          80
2653       13    108        163   55          55
3880       19    105        158   53          53


In [9]:
X_snapshot = validation_snapshot[feature_cols]
y_snapshot = validation_snapshot["RUL"]

print(
    "All selected RUL values are within the cap:",
    (y_snapshot == validation_snapshot["RUL_capped"]).all()
)

snapshot_dummy_pred = dummy_model.predict(X_snapshot)
snapshot_linear_pred = np.clip(
    linear_model.predict(X_snapshot), 0, RUL_CAP
)

snapshot_predictions = validation_snapshot[
    ["unit_id", "cycle", "RUL"]
].copy()

snapshot_predictions["dummy_prediction"] = snapshot_dummy_pred
snapshot_predictions["linear_prediction"] = snapshot_linear_pred

print("\nFirst 5 predictions:")
print(snapshot_predictions.head().round(2))

for name, predictions in [
    ("Dummy", snapshot_dummy_pred),
    ("Linear Regression", snapshot_linear_pred)
]:
    mae = mean_absolute_error(y_snapshot, predictions)
    rmse = mean_squared_error(y_snapshot, predictions) ** 0.5
    print(f"{name}: MAE={mae:.2f}, RMSE={rmse:.2f}")

All selected RUL values are within the cap: True

First 5 predictions:
      unit_id  cycle  RUL  dummy_prediction  linear_prediction
127         1    128   64             86.96              85.92
1025        5    179   90             86.96              86.79
2295       11    160   80             86.96              83.06
2653       13    108   55             86.96              72.83
3880       19    105   53             86.96              68.73
Dummy: MAE=19.37, RMSE=23.36
Linear Regression: MAE=16.85, RMSE=20.39


In [10]:
cutoff_results = []

for cutoff in [50, 100, 150, 200]:
    snapshot = val_fold.loc[val_fold["cycle"] == cutoff]
    X_snapshot = snapshot[feature_cols]
    y_snapshot = snapshot["RUL_capped"].to_numpy()

    for model_name, model in [
        ("Dummy", dummy_model),
        ("Linear Regression", linear_model)
    ]:
        predictions = np.clip(
            model.predict(X_snapshot), 0, RUL_CAP
        )
        signed_errors = predictions - y_snapshot

        cutoff_results.append({
            "cutoff_cycle": cutoff,
            "model": model_name,
            "engines": len(snapshot),
            "MAE": mean_absolute_error(y_snapshot, predictions),
            "mean_signed_error": signed_errors.mean(),
            "overestimate_pct": (signed_errors > 0).mean() * 100
        })

cutoff_comparison = pd.DataFrame(cutoff_results)
print(cutoff_comparison.round(2).to_string(index=False))

 cutoff_cycle             model  engines   MAE  mean_signed_error  overestimate_pct
           50             Dummy       20 32.09             -31.89              5.00
           50 Linear Regression       20  9.96              -7.60             20.00
          100             Dummy       20 27.51              -6.84             35.00
          100 Linear Regression       20 15.96               1.08             40.00
          150             Dummy       19 41.14              29.85             68.42
          150 Linear Regression       19 18.04               6.66             73.68
          200             Dummy        9 42.29              42.29            100.00
          200 Linear Regression        9 10.85               6.41             77.78


In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

forest_model.fit(X_train, y_train)

forest_train_pred = forest_model.predict(X_train)
forest_val_pred = forest_model.predict(X_val)

for name, actual, predicted in [
    ("Training", y_train, forest_train_pred),
    ("Validation", y_val, forest_val_pred)
]:
    print(f"\n{name}")
    print(f"MAE:  {mean_absolute_error(actual, predicted):.2f}")
    print(f"RMSE: {mean_squared_error(actual, predicted) ** 0.5:.2f}")
    print(f"R²:   {r2_score(actual, predicted):.3f}")

print(
    "\nLinear Regression validation MAE:",
    round(mean_absolute_error(y_val, val_linear_pred_clipped), 2)
)
print(
    "Random Forest prediction range:",
    round(forest_val_pred.min(), 2),
    "to",
    round(forest_val_pred.max(), 2)
)

In [11]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

forest_model.fit(X_train, y_train)

forest_train_pred = forest_model.predict(X_train)
forest_val_pred = forest_model.predict(X_val)

for name, actual, predicted in [
    ("Training", y_train, forest_train_pred),
    ("Validation", y_val, forest_val_pred)
]:
    print(f"\n{name}")
    print(f"MAE:  {mean_absolute_error(actual, predicted):.2f}")
    print(f"RMSE: {mean_squared_error(actual, predicted) ** 0.5:.2f}")
    print(f"R²:   {r2_score(actual, predicted):.3f}")

print(
    "\nLinear Regression validation MAE:",
    round(mean_absolute_error(y_val, val_linear_pred_clipped), 2)
)
print(
    "Random Forest prediction range:",
    round(forest_val_pred.min(), 2),
    "to",
    round(forest_val_pred.max(), 2)
)


Training
MAE:  7.79
RMSE: 11.63
R²:   0.922

Validation
MAE:  10.78
RMSE: 15.94
R²:   0.854

Linear Regression validation MAE: 13.14
Random Forest prediction range: 1.52 to 125.0


In [12]:
forest_cutoff_results = []

for cutoff in [50, 100, 150, 200]:
    snapshot = val_fold.loc[val_fold["cycle"] == cutoff]
    actual = snapshot["RUL_capped"].to_numpy()

    predicted = np.clip(
        forest_model.predict(snapshot[feature_cols]),
        0,
        RUL_CAP
    )

    signed_errors = predicted - actual

    forest_cutoff_results.append({
        "cutoff_cycle": cutoff,
        "model": "Random Forest",
        "engines": len(snapshot),
        "MAE": mean_absolute_error(actual, predicted),
        "mean_signed_error": signed_errors.mean(),
        "overestimate_pct": (signed_errors > 0).mean() * 100
    })

linear_cutoff_results = cutoff_comparison.loc[
    cutoff_comparison["model"] == "Linear Regression"
]

model_cutoff_comparison = pd.concat(
    [linear_cutoff_results, pd.DataFrame(forest_cutoff_results)],
    ignore_index=True
).sort_values(["cutoff_cycle", "model"])

print(model_cutoff_comparison.round(2).to_string(index=False))

 cutoff_cycle             model  engines   MAE  mean_signed_error  overestimate_pct
           50 Linear Regression       20  9.96              -7.60             20.00
           50     Random Forest       20  6.88               1.42             35.00
          100 Linear Regression       20 15.96               1.08             40.00
          100     Random Forest       20 15.93              -2.22             40.00
          150 Linear Regression       19 18.04               6.66             73.68
          150     Random Forest       19 13.03               0.52             57.89
          200 Linear Regression        9 10.85               6.41             77.78
          200     Random Forest        9 10.48              10.25             77.78


In [13]:
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

cv_results = []
group_cv = GroupKFold(n_splits=5)

for fold, (fit_idx, check_idx) in enumerate(
    group_cv.split(train, groups=train["unit_id"]),
    start=1
):
    fit_rows = train.iloc[fit_idx]
    check_rows = train.iloc[check_idx]

    # Her turda constant features yalnızca eğitim motorlarından belirlenir.
    fold_features = [
        col for col in candidate_features
        if fit_rows[col].nunique() > 1
    ]

    X_fit = fit_rows[fold_features]
    y_fit = fit_rows["RUL_capped"]
    X_check = check_rows[fold_features]
    y_check = check_rows["RUL_capped"]

    for model_name, base_model in [
        ("Linear Regression", linear_model),
        ("Random Forest", forest_model)
    ]:
        model = clone(base_model)
        model.fit(X_fit, y_fit)

        predictions = np.clip(
            model.predict(X_check), 0, RUL_CAP
        )

        cv_results.append({
            "fold": fold,
            "model": model_name,
            "engines": check_rows["unit_id"].nunique(),
            "MAE": mean_absolute_error(y_check, predictions),
            "RMSE": mean_squared_error(y_check, predictions) ** 0.5
        })

    print(f"Fold {fold}/5 completed")

cv_table = pd.DataFrame(cv_results)

print("\nResults by fold:")
print(cv_table.round(2).to_string(index=False))

print("\nMean and variation across folds:")
print(
    cv_table.groupby("model")[["MAE", "RMSE"]]
    .agg(["mean", "std"])
    .round(2)
)

Fold 1/5 completed
Fold 2/5 completed
Fold 3/5 completed
Fold 4/5 completed
Fold 5/5 completed

Results by fold:
 fold             model  engines   MAE  RMSE
    1 Linear Regression       20 17.20 21.56
    1     Random Forest       20 12.30 17.50
    2 Linear Regression       20 15.25 19.90
    2     Random Forest       20 11.14 16.48
    3 Linear Regression       20 14.60 19.56
    3     Random Forest       20 11.91 17.06
    4 Linear Regression       20 12.91 16.97
    4     Random Forest       20 10.58 15.37
    5 Linear Regression       20 14.49 19.34
    5     Random Forest       20 11.16 17.03

Mean and variation across folds:
                     MAE         RMSE      
                    mean   std   mean   std
model                                      
Linear Regression  14.89  1.55  19.47  1.64
Random Forest      11.42  0.68  16.69  0.82


In [14]:
near_failure_predictions = []
group_cv = GroupKFold(n_splits=5)

for fold, (fit_idx, check_idx) in enumerate(
    group_cv.split(train, groups=train["unit_id"]),
    start=1
):
    fit_rows = train.iloc[fit_idx]
    check_rows = train.iloc[check_idx]

    fold_features = [
        col for col in candidate_features
        if fit_rows[col].nunique() > 1
    ]

    near_check = check_rows.loc[check_rows["RUL"] <= 50]
    actual = near_check["RUL"].to_numpy()

    for model_name, base_model in [
        ("Linear Regression", linear_model),
        ("Random Forest", forest_model)
    ]:
        model = clone(base_model)
        model.fit(
            fit_rows[fold_features],
            fit_rows["RUL_capped"]
        )

        predicted = np.clip(
            model.predict(near_check[fold_features]),
            0,
            RUL_CAP
        )

        near_failure_predictions.append(pd.DataFrame({
            "fold": fold,
            "model": model_name,
            "unit_id": near_check["unit_id"].to_numpy(),
            "RUL": actual,
            "prediction": predicted
        }))

    print(f"Fold {fold}/5 completed")

near_oof = pd.concat(
    near_failure_predictions, ignore_index=True
)

near_oof["signed_error"] = (
    near_oof["prediction"] - near_oof["RUL"]
)
near_oof["absolute_error"] = near_oof["signed_error"].abs()
near_oof["overestimated"] = near_oof["signed_error"] > 0

near_oof["RUL_range"] = pd.cut(
    near_oof["RUL"],
    bins=[-1, 25, 50],
    labels=["0–25", "26–50"]
)

near_summary = near_oof.groupby(
    ["model", "RUL_range"], observed=True
).agg(
    engines=("unit_id", "nunique"),
    rows=("RUL", "size"),
    MAE=("absolute_error", "mean"),
    mean_signed_error=("signed_error", "mean"),
    overestimate_pct=("overestimated", "mean")
)

near_summary["overestimate_pct"] *= 100

print("\nNear-failure out-of-fold results:")
print(near_summary.round(2))

Fold 1/5 completed
Fold 2/5 completed
Fold 3/5 completed
Fold 4/5 completed
Fold 5/5 completed

Near-failure out-of-fold results:
                             engines  rows    MAE  mean_signed_error  \
model             RUL_range                                            
Linear Regression 0–25           100  2600  13.92              10.34   
                  26–50          100  2500  20.34              17.18   
Random Forest     0–25           100  2600   5.79               4.07   
                  26–50          100  2500  14.95              10.37   

                             overestimate_pct  
model             RUL_range                    
Linear Regression 0–25                  69.46  
                  26–50                 86.36  
Random Forest     0–25                  74.19  
                  26–50                 66.56  


In [16]:
from sklearn.base import clone

# Train verisinde değişen özellikleri seç
full_feature_cols = [
    col for col in candidate_features
    if train[col].nunique() > 1
]

# Seçtiğimiz modeli tüm train motorlarıyla yeniden eğit
final_forest = clone(forest_model)
final_forest.fit(train[full_feature_cols], train["RUL_capped"])

# Her test motorunun yalnızca son gözlemini al
last_test = (
    test.sort_values(["unit_id", "cycle"])
        .groupby("unit_id")
        .tail(1)
        .sort_values("unit_id")
        .reset_index(drop=True)
)

assert len(last_test) == len(test_rul) == 100
assert last_test["unit_id"].tolist() == list(range(1, 101))

# Motor başına bir tahmin üret
predicted_rul = final_forest.predict(last_test[full_feature_cols])

test_predictions = last_test[["unit_id", "cycle"]].copy()
test_predictions = test_predictions.rename(
    columns={"cycle": "last_observed_cycle"}
)
test_predictions["predicted_RUL_capped"] = predicted_rul

print("Prediction count:", len(test_predictions))
print("Prediction range:",
      round(test_predictions["predicted_RUL_capped"].min(), 2),
      "to",
      round(test_predictions["predicted_RUL_capped"].max(), 2))
print(test_predictions.head())

Prediction count: 100
Prediction range: 7.01 to 124.22
   unit_id  last_observed_cycle  predicted_RUL_capped
0        1                   31            124.216074
1        2                   49            121.940081
2        3                  126             55.759650
3        4                  106             75.832324
4        5                   98             93.035979


In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# RUL dosyasındaki sıra, test motorlarının unit_id sırasına karşılık geliyor.
test_predictions["true_RUL"] = test_rul.iloc[:, 0].to_numpy()
test_predictions["true_RUL_capped"] = test_predictions["true_RUL"].clip(upper=RUL_CAP)

prediction = test_predictions["predicted_RUL_capped"]

for name, target in [
    ("Actual test RUL", test_predictions["true_RUL"]),
    ("Capped test RUL", test_predictions["true_RUL_capped"]),
]:
    print(name)
    print("MAE: ", round(mean_absolute_error(target, prediction), 2))
    print("RMSE:", round(mean_squared_error(target, prediction) ** 0.5, 2))
    print("R²:  ", round(r2_score(target, prediction), 3))
    print()

print("Actual RUL above 125:",
      (test_predictions["true_RUL"] > RUL_CAP).sum())

print(test_predictions.head())

Actual test RUL
MAE:  13.51
RMSE: 18.02
R²:   0.812

Capped test RUL
MAE:  12.44
RMSE: 16.78
R²:   0.825

Actual RUL above 125: 11
   unit_id  last_observed_cycle  predicted_RUL_capped  true_RUL  \
0        1                   31            124.216074       112   
1        2                   49            121.940081        98   
2        3                  126             55.759650        69   
3        4                  106             75.832324        82   
4        5                   98             93.035979        91   

   true_RUL_capped  
0              112  
1               98  
2               69  
3               82  
4               91  


In [18]:
test_predictions["signed_error"] = (
    test_predictions["predicted_RUL_capped"] - test_predictions["true_RUL"]
)
test_predictions["absolute_error"] = test_predictions["signed_error"].abs()

errors = test_predictions["signed_error"]
columns_to_show = [
    "unit_id",
    "last_observed_cycle",
    "true_RUL",
    "predicted_RUL_capped",
    "signed_error",
]

print("Mean signed error:", round(errors.mean(), 2))
print("Overestimated engines:", (errors > 0).sum())
print("Underestimated engines:", (errors < 0).sum())

print("\n5 largest overestimations:")
print(
    test_predictions.sort_values("signed_error", ascending=False)
    [columns_to_show].head(5).round(2).to_string(index=False)
)

print("\n5 largest underestimations:")
print(
    test_predictions.sort_values("signed_error")
    [columns_to_show].head(5).round(2).to_string(index=False)
)

Mean signed error: -0.76
Overestimated engines: 52
Underestimated engines: 48

5 largest overestimations:
 unit_id  last_observed_cycle  true_RUL  predicted_RUL_capped  signed_error
      41                  123        18                 60.25         42.25
      67                   71        77                114.14         37.14
      15                   76        83                120.13         37.13
      37                  121        21                 55.19         34.19
      18                  133        28                 57.50         29.50

5 largest underestimations:
 unit_id  last_observed_cycle  true_RUL  predicted_RUL_capped  signed_error
      89                  177       136                 90.60        -45.40
      45                  152       114                 71.38        -42.62
      74                  137       126                 84.05        -41.95
      30                  143       115                 76.49        -38.51
      55                  113

In [19]:
near_test = test_predictions[test_predictions["true_RUL"] <= 50].copy()

near_test["RUL_range"] = pd.cut(
    near_test["true_RUL"],
    bins=[-1, 25, 50],
    labels=["0–25", "26–50"]
)

near_summary = (
    near_test.groupby("RUL_range", observed=True)
    .agg(
        engines=("unit_id", "count"),
        MAE=("absolute_error", "mean"),
        mean_signed_error=("signed_error", "mean"),
        overestimate_pct=(
            "signed_error",
            lambda errors: 100 * (errors > 0).mean()
        )
    )
    .round(2)
)

print("Near-failure test engines:", len(near_test))
print(near_summary)

print("\n10 engines closest to failure:")
print(
    near_test[
        ["unit_id", "true_RUL", "predicted_RUL_capped", "signed_error"]
    ]
    .sort_values("true_RUL")
    .head(10)
    .round(2)
    .to_string(index=False)
)

Near-failure test engines: 33
           engines   MAE  mean_signed_error  overestimate_pct
RUL_range                                                    
0–25            19  8.77               7.48             68.42
26–50           14  7.88               5.99             78.57

10 engines closest to failure:
 unit_id  true_RUL  predicted_RUL_capped  signed_error
      34         7                  7.01          0.01
      68         8                 18.35         10.35
      31         8                 21.30         13.30
      81         8                  7.96         -0.04
      82         9                  8.85         -0.15
      76        10                 13.02          3.02
      42        10                 20.30         10.30
      35        11                  7.46         -3.54
      66        14                 15.86          1.86
      56        15                 14.54         -0.46


## Baseline model — sonuçlar

- Hedef değişken olarak `RUL_capped = min(RUL, 125)` kullanıldı.
- Motorlar birbirinden ayrı tutularak yapılan 5-fold GroupKFold değerlendirmesinde Random Forest ortalama MAE **11.42**, Linear Regression ise **14.89** elde etti. Bu nedenle final baseline model olarak Random Forest seçildi.
- Her test motorunun son gözleminden bir tahmin üretildi (100 motor). Gerçek test RUL değerlerine göre MAE **13.51 cycle**, RMSE **18.02 cycle** ve R² **0.812**.
- Test motorlarının 11’inde gerçek RUL 125’in üzerindeydi. Capped test RUL ile hesaplanan MAE **12.44 cycle** oldu. Bu iki MAE farklı hedef tanımlarını ölçer.
- Arızaya yakın test motorlarında fazla tahmin eğilimi görüldü: gerçek RUL 0–25 olan 19 motorun %68.42’sinde, RUL 26–50 olan 14 motorun %78.57’sinde model kalan ömrü fazla tahmin etti.
- Örneğin motor 41 için gerçek RUL 18 cycle iken tahmin 60.25 cycle oldu. Bu tür hatalar bakımın geç planlanması açısından önemlidir.

GroupKFold sonucu her cycle için, test sonucu ise motor başına yalnızca son gözlem için hesaplandı; bu iki MAE doğrudan aynı değerlendirme değildir. Sonraki aşamada geçmiş sensör ölçümlerinden rolling mean ve trend özellikleri üretip bunları train verisinde GroupKFold ile değerlendireceğiz.